In [ ]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np

result_files = {
    "Popularity": "popularity_validation_results.csv",
    "Item-based CF": "item_based_validation_results.csv",
    "Genres": "genre_validation_results.csv",
    "Original TF-IDF": "tfidf_validation_results.csv",
    "Log-TF TF-IDF": "log_tfidf_validation_results.csv",
    "Embeddings": "embedding_validation_results.csv",
    "Adjusted embeddings": "adjusted_embedding_validation_results.csv"
}

evaluations = {
    model_name: (
        pd.read_csv(processed_dir / filename)
        .set_index("userId")
        .sort_index()
    )
    for model_name, filename in result_files.items()
}

In [ ]:
# This recent evaluation contains the training-history flag
reference = evaluations["Embeddings"]
history_mask = reference["has_training_history"]

assert reference.index.is_unique
assert (reference["positive_movie_count"] > 0).all()

comparison_rows = []

for model_name, results in evaluations.items():
    assert results.index.is_unique, model_name

    assert results.index.equals(reference.index), (
        f"{model_name}: evaluated users differ"
    )

    assert (
        results["positive_movie_count"]
        == reference["positive_movie_count"]
    ).all(), f"{model_name}: recall denominators differ"

    assert np.allclose(
        results["recall_at_10"],
        results["hits_at_10"] / results["positive_movie_count"]
    ), f"{model_name}: saved recall values are inconsistent"

    if "has_training_history" in results.columns:
        assert (
            results["has_training_history"] == history_mask
        ).all(), f"{model_name}: training-history flags differ"

    comparison_rows.append({
        "model": model_name,
        "users": len(results),
        "mean_recall_at_10": results["recall_at_10"].mean(),
        "recall_with_history": (
            results.loc[history_mask, "recall_at_10"].mean()
        ),
        "recall_without_history": (
            results.loc[~history_mask, "recall_at_10"].mean()
        ),
        "total_hits_at_10": results["hits_at_10"].sum()
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("mean_recall_at_10", ascending=False)
    .reset_index(drop=True)
)

display(
    model_comparison.style.format({
        "mean_recall_at_10": "{:.4%}",
        "recall_with_history": "{:.4%}",
        "recall_without_history": "{:.4%}"
    })
)

model_comparison.to_csv(
    processed_dir / "validation_model_comparison.csv",
    index=False
)

,model,users,mean_recall_at_10,recall_with_history,recall_without_history,total_hits_at_10
0,Item-based CF,165,4.4762%,6.1086%,4.1847%,326
1,Popularity,165,3.9753%,2.8027%,4.1847%,315
2,Original TF-IDF,165,3.7608%,1.3870%,4.1847%,305
3,Log-TF TF-IDF,165,3.6956%,0.9569%,4.1847%,295
4,Adjusted embeddings,165,3.5910%,0.2664%,4.1847%,297
5,Genres,165,3.5802%,0.1950%,4.1847%,295
6,Embeddings,165,3.5616%,0.0724%,4.1847%,295


Item-based collaborative filtering was selected using validation
mean Recall@10. Its positive-rating threshold (4.0), mean cosine
similarity scoring, seen-movie exclusions, tie rules, and popularity
fallback are fixed.

The final model will use training plus validation interactions.
Popularity will serve as the predefined test benchmark.

In [ ]:
raw_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(raw_dir / "movies.csv")
ratings = pd.read_csv(raw_dir / "ratings.csv")

ratings_with_movies = ratings.merge(
    movies,
    on="movieId",
    how="left",
    validate="many_to_one"
)

ratings_with_movies["rated_at"] = pd.to_datetime(
    ratings_with_movies["timestamp"],
    unit="s",
    utc=True
)

validation_end = pd.Timestamp("2016-03-22 08:26:11+00:00")

final_fit_ratings = ratings_with_movies.loc[
    ratings_with_movies["rated_at"] <= validation_end
].copy()

test_ratings = ratings_with_movies.loc[
    ratings_with_movies["rated_at"] > validation_end
].copy()

assert len(final_fit_ratings) == 80669
assert len(test_ratings) == 20167

assert (
    final_fit_ratings["rated_at"].max()
    < test_ratings["rated_at"].min()
)

In [ ]:
final_movie_popularity = (
    final_fit_ratings
    .groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        validate="one_to_one"
    )
    .sort_values(
        ["training_rating_count", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

final_feedback = final_fit_ratings[
    ["userId", "movieId", "rating"]
].copy()

final_feedback["liked"] = (
    final_feedback["rating"] >= 4.0
).astype(int)

final_user_movie_matrix = (
    final_feedback
    .pivot(
        index="userId",
        columns="movieId",
        values="liked"
    )
    .fillna(0)
    .astype(int)
)

final_movie_user_matrix = final_user_movie_matrix.T

assert set(final_movie_user_matrix.index) == set(
    final_movie_popularity["movieId"]
)

print("Final fitting ratings:", len(final_fit_ratings))
print("Test ratings:", len(test_ratings))
print("Candidate movies:", len(final_movie_popularity))
print("Movie-user matrix shape:", final_movie_user_matrix.shape)

Final fitting ratings: 80669
Test ratings: 20167
Candidate movies: 7867
Movie-user matrix shape: (7867, 522)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_popular(
    user_id,
    training_data,
    popularity_ranking,
    n=10
):
    """Return popular movies the user has not rated in training."""
    rated_movie_ids = training_data.loc[
        training_data["userId"] == user_id,
        "movieId"
    ]

    eligible_movies = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(rated_movie_ids)
    ]

    return eligible_movies.head(n).reset_index(drop=True)



def recommend_item_based(
    user_id,
    training_data,
    movie_user_matrix,
    popularity_ranking,
    n=10
):
    """Recommend using mean item similarity, with popularity as a fallback."""

    user_history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_movie_ids = user_history.loc[
        user_history["rating"] >= 4.0,
        "movieId"
    ]

    recommendations = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(user_history["movieId"])
    ].copy()

    recommendations["score"] = 0.0

    if not liked_movie_ids.empty:
        similarities = cosine_similarity(
            movie_user_matrix.loc[liked_movie_ids],
            movie_user_matrix
        )

        scores = pd.Series(
            similarities.mean(axis=0),
            index=movie_user_matrix.index
        )

        recommendations["score"] = (
            recommendations["movieId"].map(scores).fillna(0.0)
        )

    return recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    ).head(n).reset_index(drop=True)

In [ ]:
from tqdm import tqdm

test_positives = test_ratings.loc[
    test_ratings["rating"] >= 4.0
]

fitting_users = set(final_fit_ratings["userId"])
test_groups = test_positives.groupby("userId")
test_rows = []

for user_id, user_positives in tqdm(
    test_groups,
    total=test_groups.ngroups,
    desc="Evaluating test users"
):
    positive_ids = set(user_positives["movieId"])

    seen_ids = set(
        final_fit_ratings.loc[
            final_fit_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    model_recommendations = {
        "Item-based CF": recommend_item_based(
            user_id=user_id,
            training_data=final_fit_ratings,
            movie_user_matrix=final_movie_user_matrix,
            popularity_ranking=final_movie_popularity,
            n=10
        ),
        "Popularity": recommend_popular(
            user_id=user_id,
            training_data=final_fit_ratings,
            popularity_ranking=final_movie_popularity,
            n=10
        )
    }

    for model_name, recommendations in model_recommendations.items():
        recommended_ids = set(recommendations["movieId"])

        assert recommendations["movieId"].is_unique
        assert recommended_ids.isdisjoint(seen_ids)

        hits = len(positive_ids.intersection(recommended_ids))

        test_rows.append({
            "model": model_name,
            "userId": user_id,
            "positive_movie_count": len(positive_ids),
            "hits_at_10": hits,
            "recall_at_10": hits / len(positive_ids),
            "has_training_history": user_id in fitting_users
        })

test_results = pd.DataFrame(test_rows)

assert not test_results.duplicated(["model", "userId"]).any()

Evaluating test users: 100%|██████████| 116/116 [00:02<00:00, 48.22it/s]


In [ ]:
test_summary = (
    test_results
    .groupby("model", sort=False)
    .agg(
        users=("userId", "nunique"),
        mean_recall_at_10=("recall_at_10", "mean"),
        total_hits_at_10=("hits_at_10", "sum")
    )
    .reset_index()
)

test_history_summary = (
    test_results
    .groupby(["model", "has_training_history"])
    .agg(
        users=("userId", "nunique"),
        mean_recall_at_10=("recall_at_10", "mean"),
        total_hits_at_10=("hits_at_10", "sum")
    )
    .reset_index()
)

print(
    "Test users evaluated:",
    test_results["userId"].nunique()
)

display(
    test_summary.style.format({
        "mean_recall_at_10": "{:.4%}"
    })
)

display(
    test_history_summary.style.format({
        "mean_recall_at_10": "{:.4%}"
    })
)

Test users evaluated: 116


,model,users,mean_recall_at_10,total_hits_at_10
0,Item-based CF,116,6.3825%,329
1,Popularity,116,6.1055%,322


,model,has_training_history,users,mean_recall_at_10,total_hits_at_10
0,Item-based CF,False,88,6.9350%,301
1,Item-based CF,True,28,4.6458%,28
2,Popularity,False,88,6.9350%,301
3,Popularity,True,28,3.4984%,21


In [ ]:
test_results.to_csv(
    processed_dir / "final_test_results.csv",
    index=False
)

test_summary.to_csv(
    processed_dir / "test_model_comparison.csv",
    index=False
)

test_history_summary.to_csv(
    processed_dir / "test_results_by_history.csv",
    index=False
)

print("Saved final test evaluations")

Saved final test evaluations


The selected model beat popularity on the held-out test set, with a modest gain.
Test group	CF Recall@10	Popularity	Gain
All 116 users	6.3825%	6.1055%	+0.2770 percentage points
28 with history	4.6458%	3.4984%	+1.1474 percentage points
88 without history	6.9350%	6.9350%	0


CF recovered seven additional positive movies, entirely among users with fitting history.

Before writing the final conclusion, let’s estimate uncertainty around that gain. We’ll bootstrap the per-user recall differences, preserving the pairing between models, using SciPy’s bias-corrected bootstrap interval.

In [ ]:
from scipy.stats import bootstrap

paired = test_results.pivot(
    index="userId",
    columns="model",
    values="recall_at_10"
)

assert paired[["Item-based CF", "Popularity"]].notna().all().all()

paired["recall_gain"] = (
    paired["Item-based CF"] - paired["Popularity"]
)

history_flags = (
    test_results.loc[
        test_results["model"] == "Item-based CF"
    ]
    .set_index("userId")["has_training_history"]
)

paired["has_training_history"] = history_flags

groups = {
    "All test users": paired["recall_gain"],
    "With fitting history": paired.loc[
        paired["has_training_history"],
        "recall_gain"
    ]
}

bootstrap_rows = []

for group_name, differences in groups.items():
    interval = bootstrap(
        (differences.to_numpy(),),
        np.mean,
        n_resamples=10000,
        confidence_level=0.95,
        method="BCa",
        rng=np.random.default_rng(42)
    ).confidence_interval

    bootstrap_rows.append({
        "group": group_name,
        "users": len(differences),
        "mean_gain_pp": 100 * differences.mean(),
        "ci_lower_pp": 100 * interval.low,
        "ci_upper_pp": 100 * interval.high
    })

gain_uncertainty = pd.DataFrame(bootstrap_rows)

display(
    gain_uncertainty.style.format({
        "mean_gain_pp": "{:+.4f}",
        "ci_lower_pp": "{:+.4f}",
        "ci_upper_pp": "{:+.4f}"
    })
)

gain_uncertainty.to_csv(
    processed_dir / "test_recall_gain_uncertainty.csv",
    index=False
)

,group,users,mean_gain_pp,ci_lower_pp,ci_upper_pp
0,All test users,116,+0.2770,-0.6766,+1.2643
1,With fitting history,28,+1.1474,-3.0187,+5.3113
